In [1]:
# # Célula 1: Instalar dependências (caso necessário)
# !pip install rasterio geopandas scikit-learn alphashape buildregularizer

# Célula 2: Importar bibliotecas
import rasterio
import geopandas as gpd
import numpy as np
import shapely
from shapely.geometry import Point, mapping
from sklearn.cluster import DBSCAN
# import alphashape
# from buildingregulariser import regularize_geodataframe
from buildingregulariser import regularize_geodataframe
import matplotlib.pyplot as plt
# import alphashape


In [2]:
# Célula 3: Ler o polígono do GPKG
gdf = gpd.read_file("data/bela_vista.gpkg")
gdf = gdf.to_crs(epsg=31983)  # Converter para EPSG:31983
polygon = gdf.geometry

/Users/fernandogomes/miniconda3/envs/geo-clean/lib/python3.11/site-packages/pyogrio/geopandas.py:275: UserWarning: More than one layer found in 'bela_vista.gpkg': 'belavista_distrito' (default), 'reprojetadoa'. Specify layer parameter to avoid this warning.
  result = read_func(


In [3]:
from rasterio.mask import mask

# Célula 4: Recortar o raster pelo polígono
with rasterio.open("../LiDAR_produtos/2024/BHM-2024-50cm.tiff") as src:
    out_image, out_transform = mask(src, [mapping(polygon[0])], crop=True)
    out_meta = src.meta.copy()

In [4]:
# Célula 5: Extrair pontos X, Y, Z do raster recortado
rows, cols = np.where(out_image[0] != src.nodata)
xs, ys = rasterio.transform.xy(out_transform, rows, cols)
zs = out_image[0][rows, cols]
points = np.column_stack([xs, ys, zs])

In [5]:
# Célula 6: Rodar DBSCAN
db = DBSCAN(eps=3.0, min_samples=30).fit(points)
labels = db.labels_

In [6]:
n_grupos = len(set(labels)) - (1 if -1 in labels else 0)
print(f"Número de grupos encontrados: {n_grupos}")

Número de grupos encontrados: 3296


In [7]:
from shapely.geometry import MultiPoint

polygons = []
moda_z_list = []
total_labels = len(set(labels)) - (1 if -1 in labels else 0)
for i, label in enumerate(set(labels)):
    # if i >= 50:
        # break
    if label == -1:
        continue  # Ignorar ruído
    cluster_points = points[labels == label]
    cluster_geom = [Point(x, y) for x, y, z in cluster_points]
    # Calcular a moda dos valores Z do grupo
    z_values = cluster_points[:, 2]
    vals, counts = np.unique(np.round(z_values, 2), return_counts=True)
    moda_z = float(vals[np.argmax(counts)])
    moda_z_list.append(moda_z)
    multipoint = MultiPoint(cluster_geom)
    alpha_shape = shapely.concave_hull(multipoint, 0.1, allow_holes=False)
    polygons.append(alpha_shape)
    print(f"\rProcessados: {i+1} de {max(total_labels, n_grupos)}", end='')


Processados: 3296 de 3296

In [8]:
gdf_polygons = gpd.GeoDataFrame(geometry=polygons, crs=gdf.crs)
gdf_polygons['gabarito_mediano_2_5_D'] = moda_z_list

In [9]:
# gdf_polygons.plot()

In [10]:
# Regularizar os polígonos usando a função correta
regularized_polygons = regularize_geodataframe(gdf_polygons)

/Users/fernandogomes/miniconda3/envs/geo-clean/lib/python3.11/multiprocessing/pool.py:48: UserWarning: Regularized polygon has low IoU with original polygon. Returning original polygon.
  return list(map(*args))
/Users/fernandogomes/miniconda3/envs/geo-clean/lib/python3.11/multiprocessing/pool.py:48: UserWarning: Regularized polygon has low IoU with original polygon. Returning original polygon.
  return list(map(*args))
/Users/fernandogomes/miniconda3/envs/geo-clean/lib/python3.11/multiprocessing/pool.py:48: UserWarning: Regularized polygon has low IoU with original polygon. Returning original polygon.
  return list(map(*args))
/Users/fernandogomes/miniconda3/envs/geo-clean/lib/python3.11/multiprocessing/pool.py:48: UserWarning: Regularized polygon has low IoU with original polygon. Returning original polygon.
  return list(map(*args))
/Users/fernandogomes/miniconda3/envs/geo-clean/lib/python3.11/multiprocessing/pool.py:48: UserWarning: Regularized polygon has low IoU with original pol

In [11]:
# regularized_polygons.plot()

In [12]:
# grava regularized_polygons em um arquivo GPKG
regularized_polygons.to_file("data/poligonos_bela_vista_2_5_D.gpkg", driver="GPKG")